In [ ]:
import os
import json
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

print('TensorFlow version:', tf.__version__)

# Dataset Paths
dataset_path = './split_dataset/'
train_path = os.path.join(dataset_path, 'train')
val_path = os.path.join(dataset_path, 'val')
test_path = os.path.join(dataset_path, 'test')

# Image Settings
input_size = 128
batch_size_num = 32

# Debug Directory
tmp_debug_path = './tmp_debug'
os.makedirs(tmp_debug_path, exist_ok=True)

# Image Data Generators
train_datagen = ImageDataGenerator(
    rescale=1/255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1/255)
test_datagen = ImageDataGenerator(rescale=1/255)

# Load Data
train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(input_size, input_size),
    batch_size=batch_size_num,
    class_mode="binary",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    val_path,
    target_size=(input_size, input_size),
    batch_size=batch_size_num,
    class_mode="binary",
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=(input_size, input_size),
    batch_size=1,
    class_mode=None,
    shuffle=False
)

# Load EfficientNet Model
efficient_net = EfficientNetB0(
    weights='imagenet',
    input_shape=(input_size, input_size, 3),
    include_top=False,
    pooling='max'
)

# Define Model
model = Sequential([
    efficient_net,
    Dense(512, activation='relu6'),
    Dropout(0.5),
    Dense(128, activation='relu6'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Callbacks
checkpoint_filepath = './tmp_checkpoint'
os.makedirs(checkpoint_filepath, exist_ok=True)

custom_callbacks = [
    EarlyStopping(monitor='val_loss', mode='min', patience=5, verbose=1),
    ModelCheckpoint(filepath=os.path.join(checkpoint_filepath, 'best_model.h5'),
                    monitor='val_loss', mode='min', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# Train Model
num_epochs = 20
history = model.fit(
    train_generator,
    epochs=num_epochs,
    validation_data=val_generator,
    callbacks=custom_callbacks
)

# Plot Training Performance
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.legend()

plt.show()

# Load Best Model
best_model = load_model(os.path.join(checkpoint_filepath, 'best_model.h5'))

# Generate Predictions
test_generator.reset()
preds = best_model.predict(test_generator, verbose=1)

# Save Predictions
test_results = pd.DataFrame({
    "Filename": test_generator.filenames,
    "Prediction": preds.flatten()
})
test_results.to_csv('predictions.csv', index=False)
print("Predictions saved as predictions.csv ✅")